# Baseline Evaluation for Sarcasm Style Flipping

This notebook benchmarks three pre-trained seq2seq models on the sarcasm-style rewriting task before fine-tuning:

- `t5-base`
- `facebook/bart-base`
- `google/flan-t5-base`

The notebook first builds a deterministic balanced test set from `Sarcasm_Headlines_Dataset_v2.json` using roughly 10% of the data, then asks each model to flip the style for every headline in that test set:

- sarcastic -> non-sarcastic
- non-sarcastic -> sarcastic

It then evaluates each generated headline with the two metrics already implemented in this repo:

- semantic preservation via cosine similarity
- fluency via GPT-2 perplexity

Notes:

- `t5-base` and `facebook/bart-base` are not instruction-tuned, so zero-shot performance is expected to be weak. That is still useful as a pre-fine-tuning baseline.
- The sarcasm-flip classifier metric is intentionally not included yet, because you said to ignore that part for now.
- The current setup uses only the deterministic balanced 10% test split, which is much more practical for repeated baseline benchmarking on a Colab T4.


In [ ]:
# Pulling code, install dependencies

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LINCHENYU2030S/CS4248_group_project.git"
REPO_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_ROOT = Path("/content/CS4248_group_project")
LOCAL_PROJECT_EVAL_DIR = LOCAL_PROJECT_ROOT / "evaluation"
LOCAL_EVAL_DIR = Path("/content/evaluation")

if "google.colab" in sys.modules:
    current_dir = Path.cwd().resolve()
    if current_dir.name == "evaluation":
        project_eval_dir = current_dir
    elif (current_dir / "evaluation").exists():
        project_eval_dir = current_dir / "evaluation"
    elif LOCAL_PROJECT_EVAL_DIR.exists():
        project_eval_dir = LOCAL_PROJECT_EVAL_DIR
    elif LOCAL_EVAL_DIR.exists():
        project_eval_dir = LOCAL_EVAL_DIR
    else:
        if not REPO_ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
        project_eval_dir = REPO_ROOT / "evaluation"
    os.chdir(project_eval_dir)

PROJECT_EVAL_DIR = Path.cwd().resolve()
if PROJECT_EVAL_DIR.name != "evaluation":
    raise RuntimeError(f"Expected to run inside the evaluation directory, got: {PROJECT_EVAL_DIR}")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if str(PROJECT_EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_EVAL_DIR))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "transformers>=4.39.0",
        "sentence-transformers>=2.6.0",
        "accelerate>=0.28.0",
        "seaborn>=0.13.0",
        "ipywidgets>=8.1.0",
        "sentencepiece>=0.1.99",
    ],
    check=True,
)

print(f"Working directory: {PROJECT_EVAL_DIR}")


In [ ]:
# Upload dataset at run time

import shutil

DATASET_FILENAME = "Sarcasm_Headlines_Dataset_v2.json"
runtime_dataset_path = PROJECT_EVAL_DIR / DATASET_FILENAME

if runtime_dataset_path.exists():
    print(f"Dataset already available at: {runtime_dataset_path}")
elif "google.colab" in sys.modules:
    from google.colab import files

    print(f"Upload {DATASET_FILENAME} when the file picker opens.")
    uploaded = files.upload()

    if DATASET_FILENAME not in uploaded:
        raise FileNotFoundError(
            f"Expected {DATASET_FILENAME!r}. Uploaded files: {list(uploaded)}"
        )

    upload_source = Path("/") / DATASET_FILENAME
    if not upload_source.exists():
        raise FileNotFoundError(f"Uploaded file not found at {upload_source}")

    shutil.move(str(upload_source), str(runtime_dataset_path))
    print(f"Dataset saved to: {runtime_dataset_path}")
else:
    raise FileNotFoundError(
        f"Dataset not found at {runtime_dataset_path}. Place the JSON file there before continuing."
    )


In [ ]:
# set seed, set up device

import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from matplotlib import pyplot as plt
from transformers import set_seed

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_colwidth", 120)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE != "cuda":
    print("Warning: GPU is not enabled. This benchmark will be much slower on CPU.")


In [ ]:
# Define models, and other parameters for benchmarking

@dataclass(frozen=True)
class ModelSpec:
    key: str
    label: str
    hf_name: str


MODEL_SPECS = [
    ModelSpec(key="t5_base", label="T5-base", hf_name="t5-base"),
    ModelSpec(key="bart_base", label="BART-base", hf_name="facebook/bart-base"),
    ModelSpec(key="flan_t5_base", label="FLAN-T5-base", hf_name="google/flan-t5-base"),
]

DATASET_PATH = PROJECT_EVAL_DIR / DATASET_FILENAME
OUTPUT_DIR = PROJECT_EVAL_DIR / "baseline_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_SET_FRACTION = 0.10
RUN_ONLY_MODEL_KEYS = None
FORCE_REGENERATE = False
FORCE_RESCORE = False

GENERATION_BATCH_SIZE = 16
PERPLEXITY_BATCH_SIZE = 16
MAX_SOURCE_LENGTH = 64
MAX_NEW_TOKENS = 32
USE_FP16_ON_GPU = True

GENERATION_KWARGS = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "num_beams": 1,
    "early_stopping": True,
}

RUN_NAME = f"deterministic_balanced_test_{int(TEST_SET_FRACTION * 100)}pct"
ACTIVE_MODEL_SPECS = MODEL_SPECS
if RUN_ONLY_MODEL_KEYS is not None:
    ACTIVE_MODEL_SPECS = [spec for spec in MODEL_SPECS if spec.key in set(RUN_ONLY_MODEL_KEYS)]

print("Models:", [spec.hf_name for spec in ACTIVE_MODEL_SPECS])
print(f"Dataset path: {DATASET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Run name: {RUN_NAME}")
print(f"Test set fraction: {TEST_SET_FRACTION:.0%}")


In [ ]:
# Helper functions imported from utils.py

from utils import evaluate_generations, load_dataset, run_generation_for_model, summarise_results


In [ ]:
dataset_df = load_dataset(dataset_path=DATASET_PATH, test_fraction=TEST_SET_FRACTION)

display(dataset_df.head())
print(f"Number of evaluation examples: {len(dataset_df):,}")
print("Label counts:")
print(dataset_df["is_sarcastic"].value_counts().sort_index())


In [ ]:
all_results = []

for model_spec in ACTIVE_MODEL_SPECS:
    print(f"\n=== Running baseline generation for {model_spec.label} ===")
    generated_df = run_generation_for_model(
        dataset_df=dataset_df,
        model_spec=model_spec,
        output_dir=OUTPUT_DIR,
        run_name=RUN_NAME,
        device=DEVICE,
        batch_size=GENERATION_BATCH_SIZE,
        max_source_length=MAX_SOURCE_LENGTH,
        generation_kwargs=GENERATION_KWARGS,
        use_fp16_on_gpu=USE_FP16_ON_GPU,
        force_regenerate=FORCE_REGENERATE,
    )

    print(f"Scoring {model_spec.label} outputs")
    scored_df = evaluate_generations(
        generated_df=generated_df,
        output_dir=OUTPUT_DIR,
        run_name=RUN_NAME,
        perplexity_batch_size=PERPLEXITY_BATCH_SIZE,
        force_rescore=FORCE_RESCORE,
    )
    all_results.append(scored_df)

benchmark_results_df = pd.concat(all_results, ignore_index=True)
benchmark_results_path = OUTPUT_DIR / f"{RUN_NAME}_baseline_generation_metrics_all_models.csv"
benchmark_results_df.to_csv(benchmark_results_path, index=False)

print(f"Saved combined results to {benchmark_results_path}")
display(benchmark_results_df.head())


In [ ]:
summary_df = summarise_results(benchmark_results_df)
summary_path = OUTPUT_DIR / f"{RUN_NAME}_baseline_metrics_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print(f"Saved summary metrics to {summary_path}")


In [ ]:
plot_df = summary_df.copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=plot_df,
    x="model_label",
    y="mean_cosine_similarity",
    hue="model_label",
    ax=axes[0],
    palette="crest",
)
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
axes[0].set_title("Meaning Preservation")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean cosine similarity")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(
    data=plot_df,
    x="model_label",
    y="mean_perplexity",
    hue="model_label",
    ax=axes[1],
    palette="flare",
)
if axes[1].legend_ is not None:
    axes[1].legend_.remove()
axes[1].set_title("Fluency")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean perplexity (lower is better)")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
valid_results_df = benchmark_results_df.loc[
    benchmark_results_df["generated_headline"].fillna("").str.strip().ne("")
].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(
    data=valid_results_df,
    x="model_label",
    y="cosine_similarity",
    hue="model_label",
    ax=axes[0],
    palette="crest",
)
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
axes[0].set_title("Cosine Similarity Distribution")
axes[0].set_xlabel("")
axes[0].set_ylabel("Cosine similarity")
axes[0].tick_params(axis="x", rotation=15)

sns.boxplot(
    data=valid_results_df,
    x="model_label",
    y="perplexity",
    hue="model_label",
    ax=axes[1],
    palette="flare",
    showfliers=False,
)
if axes[1].legend_ is not None:
    axes[1].legend_.remove()
axes[1].set_title("Perplexity Distribution")
axes[1].set_xlabel("")
axes[1].set_ylabel("Perplexity")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
preview_rows = []
for model_label, frame in benchmark_results_df.groupby("model_label"):
    preview_rows.append(frame.sample(n=min(5, len(frame)), random_state=SEED))

preview_df = pd.concat(preview_rows, ignore_index=True)[[
    "model_label",
    "source_style",
    "target_style",
    "headline",
    "generated_headline",
    "cosine_similarity",
    "perplexity",
]]

display(preview_df)
